In [45]:
# REQUIRES, IMPORTS, ETC.
# =======================
# System
# ------
import sys, time, importlib


# Third-party libraries
# ----------------------
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# MODULES
# ========
# System modules
# --------------
from pathlib import Path
from typing import Dict

# Custom types
# ------------
import src.config.ctypes as ctp
importlib.reload(ctp)
from src.config.ctypes import CleanStep, IssueType, IssueAction

# Custom modules
# --------------
from src.config.globals import *

# Helpers functions
# -----------------
import src.helpers.validations as val
importlib.reload(val)
import src.helpers.console as csl
importlib.reload(csl)
import src.helpers.charts as cht
importlib.reload(cht)


<module 'src.helpers.charts' from '/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/helpers/charts.py'>

In [46]:
# CHECK CHANGES ONF IMPORT FILES
# ==============================
import importlib
importlib.reload(cht)
importlib.reload(val)
importlib.reload(ctp)

print(csl.__file__)
print([name for name in dir(csl)])
print("+ ")

print(cht.__file__)
print([name for name in dir(cht)])
print("+ ")

print(val.__file__)
print([name for name in dir(val)])

/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/helpers/console.py
['LineBreak', 'SectionBegin', 'SectionBreak', 'SectionEnd', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'nbsp']
+ 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/helpers/charts.py
['BarCht', 'ChartType', 'Dict', 'DifferenceCht', 'DuplicatedCht', 'HistogramCht', 'LineCht', 'PlotSalesAnalytics', 'QualityCht', 'RelationalScatterCht', 'ResidualsScatterCht', 'ScatterCht', 'SequentialScatterCht', 'StripCht', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'pd', 'plt', 'sns', 'sys', 'val']
+ 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/helpers/validations.py
['Dict', 'ValidateParams', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'pd']


In [47]:
# GLOBAL VARIABLES, PATHS
# =======================
# Current file paths
# ------------------
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR.parent / 'data').resolve()

# Global variables definition for this notebook
# ---------------------------------------------
# dataset:str = '../data/ventas_con_problemas.csv'
datasetPath: str = DATA_DIR / 'ventas_con_problemas.csv'

# Pandas dataframe to hold the dataset
dfSrc: pd.DataFrame
dfCln: pd.DataFrame
# Algorithm result
res: Dict[str, any]

In [48]:
# HELPFUL FUNCTIONS DEFINITIONS
# =============================
# On dfIss
# --------

# LogDataIssues
# -------------
# Appends a subset of affected rows into the central issues tracking DataFrame, 
# attaching metadata about the cleaning step, the specific issue type, and the 
# action performed. Preserves the original row index in 'orig_idx' for traceability.
# 
# @param pd.DataFrame pDfIss Current issues tracking DataFrame.
# @param pd.DataFrame pDfRows Subset of rows exhibiting the data quality issue.
# @param CleanStep    pStep Identifier of the pipeline step (e.g., '2_nulls').
# @param IssueType    pIssueType Specific category of the detected problem (e.g., 'null_critical').
# @param IssueAction  pAction Correction strategy applied ('deleted', 'imputed', 'corrected').
# 
# @return Dict[str, Any] Standardized response dictionary with 'bOk' and updated 'data' (pd.DataFrame) or 'err'.
# HELPFUL FUNCTIONS DEFINITIONS
# =============================
def LogDataIssues(
  pDfIss: pd.DataFrame,
  pDfRows: pd.DataFrame,
  pStep: CleanStep,
  pIssueType: IssueType,
  pAction: IssueAction
) -> Dict[str, any]:
    # Validate input parameter types before execution
    resVal: Dict[str, any] = val.ValidateParams(
        {
          'pDfIss': (pDfIss, pd.DataFrame),
          'pDfRows': (pDfRows, pd.DataFrame),
          'pStep': (pStep, str),
          'pIssueType': (pIssueType, str),
          'pAction': (pAction, str)
        }, 
        sys._getframe(0).f_code.co_name
      )
    
    if not resVal['bOk']: return resVal

    try:
      # Return current tracking DataFrame unchanged if no rows matched the issue mask
      if pDfRows.empty: return {'bOk': True, 'data': pDfIss}

      # Copy affected rows and attach audit metadata columns
      dfBatch: pd.DataFrame = pDfRows.copy()
      dfBatch['orig_idx'] = dfBatch.index
      dfBatch['step'] = pStep
      dfBatch['issue_type'] = pIssueType
      dfBatch['action'] = pAction

      # Concatenate new batch with existing issues log
      dfUpdated: pd.DataFrame = (
        dfBatch.reset_index(drop=True)
        if pDfIss.empty
        else pd.concat([pDfIss, dfBatch], ignore_index=True)
      )

      return {'bOk': True, 'data': dfUpdated}

    except Exception as e:
      return {
        'bOk': False,
        'err': {
          'code': 500,
          'whre': sys._getframe(0).f_code.co_name,
          'mssg': e.args[0]
        }
      }

In [49]:
# LOAD SOURCE DATASET (from CSV file)
# ===================================
csl.SectionBegin(f"DATASET PATH: \n{datasetPath}", 0)
csl.SectionBreak(0, 0)

try:
  dfSrc = pd.read_csv(datasetPath)
  # Copy the source dataframe to keep the original data and separate from the cleaned data
  dfCln = dfSrc.copy()
  # Initialize empty dataframe to store audit trail of identified data quality issues
  dfIss = pd.DataFrame()  
  print(f"File loaded successfully. Total of original rows: {len(dfSrc)}")
  print(f"Snapshot (5 rows) of the dataset:")
  print(dfSrc.head())
  
except FileNotFoundError:
  print(f"Error: Please, check path. \n'{datasetPath}'")
  print(f"Error: File not found.")


+ =========================================================
+ DATASET PATH: 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/activities/RA1/lesson_02/data/ventas_con_problemas.csv
+ ---------------------------------------------------------
File loaded successfully. Total of original rows: 320
Snapshot (5 rows) of the dataset:
  fecha             producto  cantidad  precio_unitario   total
0   NaN  Monitor 24 pulgadas       1.0           114.18  114.18
1   NaN      Cargador rapido       6.0            37.74  226.44
2   NaN  Monitor 24 pulgadas       5.0           193.91  969.55
3   NaN     Altavoz portatil       8.0           111.84  894.72
4   NaN      Cargador rapido       2.0            15.47   30.94


In [50]:
# DATA CLEANING, 1. INITIAL LOAD AND DATA TYPE STANDARDIZATION
# ============================================================
csl.SectionBegin(f"DATASET PATH: \n{datasetPath}", 0)
csl.SectionBreak(0, 0)

# Define column groups by expected data type for standardization
txtCols: list[str] = ['fecha', 'producto']
numCols: list[str] = ['cantidad', 'precio_unitario', 'total']
totOriginalRows: int = 0

try:
  totOriginalRows = len(dfCln)
  
  # Normalize column names by removing leading/trailing spaces and forcing lowercase
  dfCln.columns = dfCln.columns.str.strip().str.lower()
  
  # Standardize text columns: strip whitespace only on non-null records to preserve actual NaN values
  for col in txtCols:
    if col in dfCln.columns:
      bNotNullMask: pd.Series = dfCln[col].notna()
      dfCln.loc[bNotNullMask, col] = dfCln.loc[bNotNullMask, col].astype(str).str.strip()
          
  # Standardize numeric columns: enforce float64 to support NaN values before step 2 (Null handling)
  for col in numCols:
    if col in dfCln.columns:
        dfCln[col] = pd.to_numeric(dfCln[col], errors='coerce').astype('float64')

  print("File loaded and standardized successfully")
  print(f"Total of original rows: {totOriginalRows}")

  print("\nStandardized column data types:")
  csl.SectionBreak(0, 0)
  print(dfCln.dtypes)

  print("\nSnapshot of rows of the dataset:")
  print(dfCln.head(10))

except Exception as e:
  res = {
    'bOk': False,
    'err': {
      'code': 500,
      'whre': sys._getframe(0).f_code.co_name,
      'mssg': e.args[0]
    }
  }
  print(f"Error({res['err']['code']}): In {res['err']['whre']} \n{res['err']['mssg']}")


+ =========================================================
+ DATASET PATH: 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/activities/RA1/lesson_02/data/ventas_con_problemas.csv
+ ---------------------------------------------------------
File loaded and standardized successfully
Total of original rows: 320

Standardized column data types:
+ ---------------------------------------------------------
fecha                  str
producto               str
cantidad           float64
precio_unitario    float64
total              float64
dtype: object

Snapshot of rows of the dataset:
        fecha             producto  cantidad  precio_unitario   total
0         NaN  Monitor 24 pulgadas       1.0           114.18  114.18
1         NaN      Cargador rapido       6.0            37.74  226.44
2         NaN  Monitor 24 pulgadas       5.0           193.91  969.55
3         NaN     Altavoz portatil       8.0           111.84  894.72
4         NaN      Cargador rapido   

In [52]:
# DATA CLEANING, 2. NULL VALUES MANAGEMENT
# ========================================
csl.SectionBegin("STEP 2: NULL VALUES MANAGEMENT", 0)
csl.SectionBreak(0, 0)

rowsBeforeNulls: int = 0

try:
    rowsBeforeNulls = len(dfCln)

    # 1. Identify, log, and exclude rows with nulls in non-recoverable critical fields ('fecha', 'producto')
    bCritNullMask: pd.Series = dfCln['fecha'].isna() | dfCln['producto'].isna()
    res = LogDataIssues(dfIss, dfCln.loc[bCritNullMask], '2_nulls', 'null_critical', 'deleted')
    if not res['bOk']: raise Exception(res['err']['mssg'])
    dfIss = res['data']
    dfCln = dfCln.loc[~bCritNullMask].copy()

    # 2. Deterministic imputation for numeric fields using: total = cantidad * precio_unitario
    # 2.a Identify, log, and recover missing 'cantidad' where 'total' and 'precio_unitario' are valid
    bQtyNullMask: pd.Series = (
      dfCln['cantidad'].isna() & 
      dfCln['total'].notna() & 
      dfCln['precio_unitario'].notna() & 
      (dfCln['precio_unitario'] != 0)
    )
    res = LogDataIssues(dfIss, dfCln.loc[bQtyNullMask], '2_nulls', 'null_cantidad', 'imputed')
    if not res['bOk']: raise Exception(res['err']['mssg'])
    dfIss = res['data']
    dfCln.loc[bQtyNullMask, 'cantidad'] = (
      dfCln.loc[bQtyNullMask, 'total'] / dfCln.loc[bQtyNullMask, 'precio_unitario']
    ).round(0)

    # 2.b Identify, log, and recover missing 'precio_unitario' where 'total' and 'cantidad' are valid
    bPriceNullMask: pd.Series = (
      dfCln['precio_unitario'].isna() & 
      dfCln['total'].notna() & 
      dfCln['cantidad'].notna() & 
      (dfCln['cantidad'] != 0)
    )
    res = LogDataIssues(dfIss, dfCln.loc[bPriceNullMask], '2_nulls', 'null_precio_unitario', 'imputed')
    if not res['bOk']: raise Exception(res['err']['mssg'])
    dfIss = res['data']
    dfCln.loc[bPriceNullMask, 'precio_unitario'] = (
      dfCln.loc[bPriceNullMask, 'total'] / dfCln.loc[bPriceNullMask, 'cantidad']
    ).round(2)

    # 2.c Identify, log, and recover missing 'total' where 'cantidad' and 'precio_unitario' are valid
    bTotalNullMask: pd.Series = (
      dfCln['total'].isna() & 
      dfCln['cantidad'].notna() & 
      dfCln['precio_unitario'].notna()
    )
    res = LogDataIssues(dfIss, dfCln.loc[bTotalNullMask], '2_nulls', 'null_total', 'imputed')
    if not res['bOk']: raise Exception(res['err']['mssg'])
    dfIss = res['data']
    dfCln.loc[bTotalNullMask, 'total'] = (
      dfCln.loc[bTotalNullMask, 'cantidad'] * dfCln.loc[bTotalNullMask, 'precio_unitario']
    ).round(2)

    # 3. Fallback exclusion: log and drop any remaining rows if multiple numeric fields were null simultaneously
    bRemNullMask: pd.Series = dfCln.isna().any(axis=1)
    if bRemNullMask.any():
      res = LogDataIssues(dfIss, dfCln.loc[bRemNullMask], '2_nulls', 'null_multiple_numeric', 'deleted')
      if not res['bOk']: raise Exception(res['err']['mssg'])
      dfIss = res['data']
      dfCln = dfCln.loc[~bRemNullMask].copy()

    # Display summary from the audit DataFrame for Step 2
    dfStepIss: pd.DataFrame = dfIss[dfIss['step'] == '2_nulls']
    print("Null values processed successfully")
    print(f"Total null issues logged: {len(dfStepIss)}")
    print("\nBreakdown by issue type and action:")
    csl.SectionBreak(0, 0)
    print(dfStepIss.groupby(['issue_type', 'action']).size())
    print(f"\nRemaining nulls in cleaned dataset: {int(dfCln.isna().sum().sum())}")
    print(f"Rows before: {rowsBeforeNulls} | Rows after: {len(dfCln)}")

except Exception as e:
    res = {
      'bOk': False,
      'err': {
        'code': 500,
        'whre': sys._getframe(0).f_code.co_name,
        'mssg': e.args[0]
      }
    }
    print(f"Error({res['err']['code']}): In {res['err']['whre']} \n{res['err']['mssg']}")


+ =========================================================
+ STEP 2: NULL VALUES MANAGEMENT
+ ---------------------------------------------------------
Null values processed successfully
Total null issues logged: 25

Breakdown by issue type and action:
+ ---------------------------------------------------------
issue_type            action 
null_cantidad         imputed     5
null_critical         deleted    10
null_precio_unitario  imputed     5
null_total            imputed     5
dtype: int64

Remaining nulls in cleaned dataset: 0
Rows before: 310 | Rows after: 310


In [ ]:
# DATA CLEANING, 3. DUPLICATE ROWS REMOVAL
# ========================================
csl.SectionBegin("STEP 3: DUPLICATE ROWS REMOVAL", 0)
csl.SectionBreak(0, 0)

rowsBeforeDupl: int = 0
keyCols: list[str] = ['fecha', 'producto', 'cantidad', 'precio_unitario', 'total']

try:
  rowsBeforeDupl = len(dfCln)

  # Identify redundant duplicate rows keeping the first valid occurrence ('keep=first')
  bDuplMask: pd.Series = dfCln.duplicated(subset=keyCols, keep='first')

  # Log the 12 redundant duplicate rows into the audit DataFrame before removal
  res = LogDataIssues(dfIss, dfCln.loc[bDuplMask], '3_duplicates', 'duplicate_row', 'deleted')
  if not res['bOk']: raise Exception(res['err']['mssg'])
  dfIss = res['data']

  # Remove duplicate rows from the working clean DataFrame
  dfCln = dfCln.loc[~bDuplMask].copy()

  # Display summary from the audit DataFrame for Step 3
  dfStepIss: pd.DataFrame = dfIss[dfIss['step'] == '3_duplicates']
  print("Duplicate rows processed successfully")
  print(f"Total duplicate rows logged and removed: {len(dfStepIss)}")
  print("\nBreakdown by issue type and action:")
  csl.SectionBreak(0, 0)
  print(dfStepIss.groupby(['issue_type', 'action']).size())
  print(f"\nRemaining duplicates in dfCln: {int(dfCln.duplicated(subset=keyCols).sum())}")
  print(f"Rows before: {rowsBeforeDupl} | Rows after: {len(dfCln)}")

except Exception as e:
  res = {
    'bOk': False,
    'err': {
      'code': 500,
      'whre': sys._getframe(0).f_code.co_name,
      'mssg': e.args[0]
    }
  }
  print(f"Error({res['err']['code']}): In {res['err']['whre']} \n{res['err']['mssg']}")
